In [ ]:
from dataclasses import asdict

import cv2
import numpy as np
import matplotlib.pyplot as plt

from uavcalibration.datasets import *
from uavcalibration.map import *
import uavcalibration.calibrate as calibrate
from uavcalibration.calibrate_pipeline import *

In [ ]:
test_size = 50
input_maxsize = 10
tolerance = 5.0

dataset = VisLocDataset(r"..\datasets\UAV_VisLoc_example")
satellite_map = GeoTiffMap([i.image_path for i in dataset.satellite_infos.values()])

async with satellite_map:
    # warm up
    uavdata = dataset[0]
    calibrator = calibrate.Calibrator(satellite_map, uavdata)
    print(await calibrator.calibrate(tolerance, plot=False))

In [ ]:

stages: list[Stage] = [
    Stage1(input_maxize=input_maxsize),
    Stage2(input_maxize=input_maxsize, map=satellite_map),
    Stage3(input_maxize=input_maxsize),
    Stage4(input_maxize=input_maxsize, tolerance=tolerance),
    Printer(input_maxize=input_maxsize),
]
pipeline = Pipeline.from_stages(stages)
async with satellite_map, pipeline:
    for i in range(test_size):
        uavdata = dataset[i]
        ctx = CalibrateCTX(uavdata)
        await pipeline.add_input(ctx)

In [ ]:
async with satellite_map, pipeline:
    for i in range(test_size):
        uavdata = dataset[i]
        calibrator = calibrate.Calibrator(satellite_map, uavdata)
        print(await calibrator.calibrate(tolerance, plot=False))